# Beat Tracker and Tempo Estimation
- Asa Picton, Connor Richardson, Dylan Baecker
- CSC475

### 1. Import libraries

In [8]:

from pathlib import Path
from joblib import Parallel, delayed
import joblib
import numpy as np
import pandas as pd
import librosa as lb
from tqdm.auto import tqdm

import scipy.signal
import matplotlib.pyplot as plt

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score, mean_absolute_error
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor



### 2. Load audio and tempo data

In [9]:
audio_path = Path("giantsteps-tempo-dataset/audio")
label_path = Path("giantsteps-tempo-dataset/annotations/tempo")

# Load tempo labels
tempo_labels = {}
for file in label_path.iterdir():
    tempo_labels[file.stem] = float(file.read_text().strip())

# Load dataset
dataset = []
for file in audio_path.iterdir():
    track_id = file.stem
    if track_id in tempo_labels:
        dataset.append((file, tempo_labels[track_id]))

print(f"Loaded {len(dataset)} audio tracks with tempo labels")


Loaded 664 audio tracks with tempo labels


In [10]:
print(len(dataset))
print(dataset[0])

664
(PosixPath('giantsteps-tempo-dataset/audio/3226172.LOFI.mp3'), 69.0)


### 3. Feature Extraction

In [ ]:
def extract_features(y_audio, sr):

    features = []

    # =========================
    # 1. Onset strength
    # =========================
    onset_env = lb.onset.onset_strength(y=y_audio, sr=sr)

    if onset_env.max() > 0:
        onset_env = onset_env / onset_env.max()

    features.extend([
        np.mean(onset_env),
        np.std(onset_env),
        np.max(onset_env),
        np.median(onset_env)
    ])

    # =========================
    # 2. Librosa tempo estimate
    # =========================
    tempo_est = lb.feature.tempo(onset_envelope=onset_env, sr=sr)[0]
    features.append(tempo_est)

    # =========================
    # 3. Autocorrelation
    # =========================
    ac = lb.autocorrelate(onset_env)
    ac = ac[:len(ac)//2]

    if ac.max() > 0:
        ac = ac / ac.max()

    # Downsample to fixed size (50 dims)
    ac_ds = np.interp(
        np.linspace(0, len(ac), 50),
        np.arange(len(ac)),
        ac
    )

    features.extend(ac_ds)

    # =========================
    # 4. Tempogram full shape
    # =========================
    tg = lb.feature.tempogram(onset_envelope=onset_env, sr=sr)
    tg_mean = np.mean(tg, axis=1)

    if tg_mean.max() > 0:
        tg_mean = tg_mean / tg_mean.max()

    # Downsample to 60 dims
    tg_ds = np.interp(
        np.linspace(0, len(tg_mean), 60),
        np.arange(len(tg_mean)),
        tg_mean
    )

    features.extend(tg_ds)

    # =========================
    # 5. Tempogram peaks 
    # =========================
    tempo_freqs = lb.tempo_frequencies(tg.shape[0], sr=sr)

    peaks, props = scipy.signal.find_peaks(tg_mean, height=0.1)

    peak_bpms = np.zeros(5)
    peak_strengths = np.zeros(5)

    if len(peaks) > 0:
        order = np.argsort(props["peak_heights"])[::-1]
        peaks = peaks[order][:5]

        selected_bpms = tempo_freqs[peaks]
        selected_strengths = tg_mean[peaks]

        peak_bpms[:len(selected_bpms)] = selected_bpms
        peak_strengths[:len(selected_strengths)] = selected_strengths

    features.extend(peak_bpms)
    features.extend(peak_strengths)

    # =========================
    # 6. Rhythm regularity features
    # =========================
    # Inter-onset interval approximation
    onset_peaks = scipy.signal.find_peaks(onset_env, height=0.3)[0]

    if len(onset_peaks) > 1:
        iois = np.diff(onset_peaks)

        features.extend([
            np.mean(iois),
            np.std(iois),
            np.median(iois)
        ])
    else:
        features.extend([0, 0, 0])

    # =========================
    # 7. Spectral flux (rhythmic energy change)
    # =========================
    S = np.abs(lb.stft(y_audio))
    flux = np.sqrt(np.sum(np.diff(S, axis=1)**2, axis=0))

    if flux.max() > 0:
        flux = flux / flux.max()

    features.extend([
        np.mean(flux),
        np.std(flux),
        np.max(flux)
    ])

    return np.array(features)

In [12]:
def process_track(path, bpm):
    y_audio, sr = lb.load(path, duration=30)
    features = extract_features(y_audio, sr)

    return features, bpm


In [13]:
CACHE_FILE = Path("features_cache.pkl")

if CACHE_FILE.exists():
    print("Loading features from cache")
    X, y = joblib.load(CACHE_FILE)
    print(f"Loaded {len(X)} cached feature vectors.")
else:
    print("Extracting features")

    results = Parallel(n_jobs=-1)(
        delayed(process_track)(path, bpm)
        for path, bpm in tqdm(dataset, desc="Tracks")
    )
    X, y = zip(*results)

    joblib.dump((X, y), CACHE_FILE)
    print(f"Extracted features for {len(X)} tracks. Saved to {CACHE_FILE}.")

Loading features from cache
Loaded 664 cached feature vectors.


In [14]:
# Convert features to numpy arrays
X = np.array(X)  
y = np.array(y) 

print(f"X shape: {X.shape}")  
print(f"y shape: {y.shape}")  

X shape: (664, 131)
y shape: (664,)


### 4. Model Training 

##### 4.1 Random Forest Regressor

In [18]:
def half_double_corrected_mae(y_true, y_pred):
    candidates = np.vstack([y_pred, y_pred*2, y_pred/2])
    errors = np.abs(candidates - y_true)

    return np.mean(np.min(errors, axis=0))


def acc1(y_true, y_pred, tol=0.04):
    return np.mean(np.abs(y_pred - y_true) <= tol * y_true)


def acc2(y_true, y_pred, tol=0.04):
    candidates = np.vstack([y_pred, y_pred*2, y_pred/2])
    errors = np.abs(candidates - y_true)

    min_errors = np.min(errors, axis=0)

    return np.mean(min_errors <= tol * y_true)



def training_loop(X, y):

    kf = KFold(n_splits=5, shuffle=True, random_state=42)

    all_mae = []
    all_acc1 = []
    all_acc2 = []

    def objective(trial, X_train, y_train):

        params = {
            "n_estimators": trial.suggest_int("n_estimators", 100, 500, step=50),
            "max_depth": trial.suggest_int("max_depth", 5, 30),
            "min_samples_split": trial.suggest_int("min_samples_split", 2, 15),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
            "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
            "bootstrap": trial.suggest_categorical("bootstrap", [True, False]),
            "min_impurity_decrease": trial.suggest_float("min_impurity_decrease", 0.0, 0.1),
        }

        model = RandomForestRegressor(
            **params,
            random_state=42,
            n_jobs=-1
        )

        scores = cross_val_score(
            model,
            X_train,
            y_train,
            cv=3,
            scoring="neg_mean_absolute_error",
            n_jobs=1
        )

        return scores.mean()

    for fold, (train_idx, test_idx) in enumerate(kf.split(X)):

        print(f"\nFold {fold+1}")

        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]


        study = optuna.create_study(direction="maximize")

        study.optimize(
            lambda trial: objective(trial, X_train, y_train),
            n_trials=25,
            show_progress_bar=True
        )

        best_params = study.best_params

        print(f"\nFold {fold+1} best params: {best_params}")
        print(f"Best CV score: {study.best_value:.3f}")

        model = RandomForestRegressor(
            **best_params,
            random_state=42,
            n_jobs=-1
        )

        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)

    
    
        errors = np.abs(y_pred - y_test)
        print(f"Errors > 30 BPM: {np.sum(errors > 30)} / {len(errors)}")
        print(f"Errors > 60 BPM: {np.sum(errors > 60)} / {len(errors)}")

        mae = half_double_corrected_mae(y_test, y_pred)
        acc1_score = acc1(y_test, y_pred)
        acc2_score = acc2(y_test, y_pred)

        print("\nFold Results")
        print(f"MAE (half/double corrected): {mae:.2f} BPM")
        print(f"Acc1 (±4% strict): {acc1_score:.3f}")
        print(f"Acc2 (±4% half/double): {acc2_score:.3f}")

        all_mae.append(mae)
        all_acc1.append(acc1_score)
        all_acc2.append(acc2_score)

    print("\nResults")

    print(f"MAE (half/double corrected): {np.mean(all_mae):.2f} ± {np.std(all_mae):.2f} BPM")
    print(f"Acc1 (±4% strict): {np.mean(all_acc1):.3f} ± {np.std(all_acc1):.3f}")
    print(f"Acc2 (±4% half/double): {np.mean(all_acc2):.3f} ± {np.std(all_acc2):.3f}")


training_loop(X, y)


Fold 1


Best trial: 24. Best value: -16.6966: 100%|██████████| 25/25 [01:06<00:00,  2.65s/it]



Fold 1 best params: {'n_estimators': 200, 'max_depth': 19, 'min_samples_split': 7, 'min_samples_leaf': 5, 'max_features': None, 'bootstrap': True, 'min_impurity_decrease': 0.056694665526574345}
Best CV score: -16.697
Errors > 30 BPM: 38 / 133
Errors > 60 BPM: 8 / 133

Fold Results
MAE (half/double corrected): 12.15 BPM
Acc1 (±4% strict): 0.451
Acc2 (±4% half/double): 0.474

Fold 2


Best trial: 24. Best value: -17.5222: 100%|██████████| 25/25 [00:49<00:00,  1.97s/it]



Fold 2 best params: {'n_estimators': 150, 'max_depth': 30, 'min_samples_split': 7, 'min_samples_leaf': 5, 'max_features': None, 'bootstrap': True, 'min_impurity_decrease': 0.009544478041900775}
Best CV score: -17.522
Errors > 30 BPM: 32 / 133
Errors > 60 BPM: 7 / 133

Fold Results
MAE (half/double corrected): 11.94 BPM
Acc1 (±4% strict): 0.436
Acc2 (±4% half/double): 0.444

Fold 3


...


Fold 3 best params: {'n_estimators': 200, 'max_depth': 22, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 'log2', 'bootstrap': False, 'min_impurity_decrease': 0.05005062364790178}
Best CV score: -17.824
Errors > 30 BPM: 27 / 133
Errors > 60 BPM: 4 / 133

Fold Results
MAE (half/double corrected): 11.33 BPM
Acc1 (±4% strict): 0.459
Acc2 (±4% half/double): 0.466

Fold 4


Best trial: 23. Best value: -17.7475: 100%|██████████| 25/25 [01:09<00:00,  2.79s/it]



Fold 4 best params: {'n_estimators': 350, 'max_depth': 12, 'min_samples_split': 5, 'min_samples_leaf': 3, 'max_features': None, 'bootstrap': True, 'min_impurity_decrease': 0.06497175229043879}
Best CV score: -17.748
Errors > 30 BPM: 24 / 133
Errors > 60 BPM: 4 / 133

Fold Results
MAE (half/double corrected): 11.89 BPM
Acc1 (±4% strict): 0.414
Acc2 (±4% half/double): 0.414

Fold 5


Best trial: 21. Best value: -16.6471: 100%|██████████| 25/25 [00:42<00:00,  1.69s/it]



Fold 5 best params: {'n_estimators': 350, 'max_depth': 13, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 'log2', 'bootstrap': False, 'min_impurity_decrease': 0.027692215809432163}
Best CV score: -16.647
Errors > 30 BPM: 37 / 132
Errors > 60 BPM: 5 / 132

Fold Results
MAE (half/double corrected): 13.30 BPM
Acc1 (±4% strict): 0.356
Acc2 (±4% half/double): 0.371

Results
MAE (half/double corrected): 12.12 ± 0.65 BPM
Acc1 (±4% strict): 0.423 ± 0.037
Acc2 (±4% half/double): 0.434 ± 0.038


In [19]:
def xgb_training_loop(X, y):

    kf = KFold(n_splits=5, shuffle=True, random_state=42)

    all_mae = []
    all_acc1 = []
    all_acc2 = []

    def objective(trial, X_train, y_train):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 200, 600),
            "max_depth": trial.suggest_int("max_depth", 3, 10),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
            "gamma": trial.suggest_float("gamma", 0, 5),
            "reg_alpha": trial.suggest_float("reg_alpha", 0, 5),
            "reg_lambda": trial.suggest_float("reg_lambda", 0.1, 10),
        }

        # Use inner cross-val instead of fitting and evaluating on the same data
        model = XGBRegressor(
            **params,
            objective="reg:squarederror",
            tree_method="hist",
            random_state=42,
            n_jobs=-1
        )

        scores = cross_val_score(
            model,
            X_train,
            y_train,
            cv=3,
            scoring="neg_mean_absolute_error",
            n_jobs=1
        )

        return scores.mean() 

    for fold, (train_idx, test_idx) in enumerate(kf.split(X)):

        print(f"\nFold {fold+1}")

        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        study = optuna.create_study(direction="maximize")

        study.optimize(
            lambda trial: objective(trial, X_train, y_train),
            n_trials=25,
            show_progress_bar=True
        )

        best_params = study.best_params

        print(f"\nFold {fold+1} best params: {best_params}")
        print(f"Best CV score: {study.best_value:.3f}")

        model = XGBRegressor(
            **best_params,
            objective="reg:squarederror",
            tree_method="hist",
            random_state=42,
            n_jobs=-1
        )

        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)

        errors = np.abs(y_pred - y_test)
        print(f"Errors > 30 BPM: {np.sum(errors > 30)} / {len(errors)}")
        print(f"Errors > 60 BPM: {np.sum(errors > 60)} / {len(errors)}")

        mae = half_double_corrected_mae(y_test, y_pred)
        acc1_score = acc1(y_test, y_pred)
        acc2_score = acc2(y_test, y_pred)

        print("\nFold Results")
        print(f"MAE (half/double corrected): {mae:.2f} BPM")
        print(f"Acc1 (±4% strict): {acc1_score:.3f}")
        print(f"Acc2 (±4% half/double): {acc2_score:.3f}")

        all_mae.append(mae)
        all_acc1.append(acc1_score)
        all_acc2.append(acc2_score)

    print("\nResults")
    print(f"MAE (half/double corrected): {np.mean(all_mae):.2f} ± {np.std(all_mae):.2f} BPM")
    print(f"Acc1 (±4% strict): {np.mean(all_acc1):.3f} ± {np.std(all_acc1):.3f}")
    print(f"Acc2 (±4% half/double): {np.mean(all_acc2):.3f} ± {np.std(all_acc2):.3f}")

xgb_training_loop(X, y)


Fold 1


Best trial: 16. Best value: -16.4207: 100%|██████████| 25/25 [03:26<00:00,  8.26s/it]



Fold 1 best params: {'n_estimators': 421, 'max_depth': 10, 'learning_rate': 0.017665158666358587, 'subsample': 0.8634647470834996, 'colsample_bytree': 0.6550071107525953, 'gamma': 4.59884235604786, 'reg_alpha': 4.313138116061814, 'reg_lambda': 4.2540852561965545}
Best CV score: -16.421
Errors > 30 BPM: 34 / 133
Errors > 60 BPM: 10 / 133

Fold Results
MAE (half/double corrected): 12.25 BPM
Acc1 (±4% strict): 0.421
Acc2 (±4% half/double): 0.436

Fold 2


Best trial: 20. Best value: -17.4558: 100%|██████████| 25/25 [03:48<00:00,  9.15s/it]



Fold 2 best params: {'n_estimators': 534, 'max_depth': 7, 'learning_rate': 0.02833045201915465, 'subsample': 0.6608838225639679, 'colsample_bytree': 0.870903648285164, 'gamma': 0.09565971954086194, 'reg_alpha': 2.286179718005267, 'reg_lambda': 6.323878158635827}
Best CV score: -17.456
Errors > 30 BPM: 29 / 133
Errors > 60 BPM: 4 / 133

Fold Results
MAE (half/double corrected): 12.87 BPM
Acc1 (±4% strict): 0.346
Acc2 (±4% half/double): 0.346

Fold 3


...


Fold 3 best params: {'n_estimators': 204, 'max_depth': 6, 'learning_rate': 0.029894145645763995, 'subsample': 0.7354241087696829, 'colsample_bytree': 0.7254592496571362, 'gamma': 3.8745596488603686, 'reg_alpha': 1.8338330831929688, 'reg_lambda': 8.387234901655244}
Best CV score: -17.684
Errors > 30 BPM: 21 / 133
Errors > 60 BPM: 3 / 133

Fold Results
MAE (half/double corrected): 10.38 BPM
Acc1 (±4% strict): 0.436
Acc2 (±4% half/double): 0.436

Fold 4


Best trial: 14. Best value: -17.5792: 100%|██████████| 25/25 [02:19<00:00,  5.60s/it]



Fold 4 best params: {'n_estimators': 276, 'max_depth': 5, 'learning_rate': 0.018122038912611, 'subsample': 0.6515163118538508, 'colsample_bytree': 0.7460131858270976, 'gamma': 0.8297425485619794, 'reg_alpha': 3.9039617027286284, 'reg_lambda': 4.9309385593546144}
Best CV score: -17.579
Errors > 30 BPM: 23 / 133
Errors > 60 BPM: 4 / 133

Fold Results
MAE (half/double corrected): 12.27 BPM
Acc1 (±4% strict): 0.361
Acc2 (±4% half/double): 0.368

Fold 5


Best trial: 10. Best value: -16.462: 100%|██████████| 25/25 [02:35<00:00,  6.22s/it]



Fold 5 best params: {'n_estimators': 341, 'max_depth': 6, 'learning_rate': 0.01986252591615231, 'subsample': 0.6033790396464366, 'colsample_bytree': 0.6154908014556691, 'gamma': 3.380930827238632, 'reg_alpha': 4.666276479122299, 'reg_lambda': 9.65968762794101}
Best CV score: -16.462
Errors > 30 BPM: 30 / 132
Errors > 60 BPM: 5 / 132

Fold Results
MAE (half/double corrected): 13.26 BPM
Acc1 (±4% strict): 0.348
Acc2 (±4% half/double): 0.371

Results
MAE (half/double corrected): 12.20 ± 0.99 BPM
Acc1 (±4% strict): 0.382 ± 0.038
Acc2 (±4% half/double): 0.392 ± 0.037
